In [ ]:
import pandas as pd
import calendar
import re

indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"
bom_file    = r"D:/Tushar/main_with_subs_only.xlsx"

# Read files
indent_df = pd.read_excel(indent_file)
bom_df    = pd.read_excel(bom_file)

# Force column names to clean strings
indent_df.columns = [str(c).strip() for c in indent_df.columns]
bom_df.columns    = [str(c).strip() for c in bom_df.columns]

# Safe string cleaner
def safe_str(s):
    return str(s).strip().replace('nan', '').replace('NaN', '')

bom_df['Main_Label'] = bom_df['Main_Label'].apply(safe_str)
bom_df['Sub_Label']  = bom_df['Sub_Label'].apply(safe_str)

# Month detection
pattern = re.compile(r"([A-Za-z]{3})'(\d{2})", re.I)
month_cols = [c for c in indent_df.columns if isinstance(c, str) and pattern.search(c)]

if not month_cols:
    print("No month-like columns found. All columns:", list(indent_df.columns))
    print("→ You may need to set latest_col manually")
    latest_col = input("Enter the exact month column name: ").strip()
else:
    latest_col = max(month_cols, key=str)
    print("Detected month column:", latest_col)

match = pattern.search(latest_col)
month_str = match.group(1).title()
year = 2000 + int(match.group(2))
month_num = list(calendar.month_abbr).index(month_str)
days = calendar.monthrange(year, month_num)[1]
print(f"Using {days} days for {month_str} {year}")

# Indent prep
part_col = 'Part number'  # ← change this if it's wrong
if part_col not in indent_df.columns:
    print("Part column not found. Possible:", [c for c in indent_df.columns if 'part' in str(c).lower() or 'code' in str(c).lower()])
    part_col = input("Enter correct switch/part column name: ").strip()

indent_df[part_col] = indent_df[part_col].apply(safe_str)
indent_df = indent_df[[part_col, latest_col]].dropna(subset=[latest_col])
indent_df[latest_col] = pd.to_numeric(indent_df[latest_col], errors='coerce').fillna(0)
indent_df = indent_df.rename(columns={part_col: 'Switch', latest_col: 'Monthly'})
indent_df['Daily'] = indent_df['Monthly'] / days

print("\nIndent preview:")
print(indent_df.head(10))

# BOM prep
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']].copy()
bom_df = bom_df.rename(columns={'Main_Label':'Child', 'Sub_Label':'Switch', 'Sub_Count':'QtyPer'})
bom_df['Switch'] = bom_df['Switch'].apply(safe_str)
bom_df['Child']  = bom_df['Child'].apply(safe_str)
bom_df['QtyPer'] = pd.to_numeric(bom_df['QtyPer'], errors='coerce').fillna(0)

print("\nBOM preview:")
print(bom_df.head(10))

# Merge
merged = bom_df.merge(indent_df[['Switch', 'Daily']], on='Switch', how='left')
merged['Daily_Need'] = merged['Daily'] * merged['QtyPer']
merged['Daily_Need'] = merged['Daily_Need'].fillna(0)

print("\nMerge rows:", len(merged))
print("Rows with match (Daily > 0):", (merged['Daily'] > 0).sum())

if (merged['Daily'] > 0).sum() == 0:
    print("\nNo matches — sample switches:")
    print("From BOM:", sorted(bom_df['Switch'].unique())[:12])
    print("From Indent:", sorted(indent_df['Switch'].unique())[:12])

# Aggregate
result = merged.groupby('Child', as_index=False)['Daily_Need'].sum()
result = result.rename(columns={'Daily_Need': 'Daily_Req'})
result['Two_Day_Req'] = result['Daily_Req'] * 2
result = result.sort_values('Two_Day_Req', ascending=False).round(2)

print("\nTop 20 results:")
print(result.head(20))

# Save
result.to_excel("2_Day_Requirement_Child_Parts.xlsx", index=False)
print("\nSaved to: 2_Day_Requirement_Child_Parts.xlsx")